In [0]:
%sql
Create database if not exists Sample

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("lab4v2").getOrCreate()

spark.sql("CREATE DATABASE IF NOT EXISTS Sample")



Out[19]: DataFrame[]

In [0]:
%sql

CREATE TABLE IF NOT EXISTS Sample.Transactions ( AccountId INT, TranDate DATE, TranAmt DECIMAL(8, 2));

CREATE TABLE IF NOT EXISTS Sample.Logical (RowID INT,FName VARCHAR(20), Salary SMALLINT);

In [0]:
%sql

INSERT INTO Sample.Transactions VALUES 
( 1, '2011-01-01', 500),
( 1, '2011-01-15', 50),
( 1, '2011-01-22', 250),
( 1, '2011-01-24', 75),
( 1, '2011-01-26', 125),
( 1, '2011-01-28', 175),
( 2, '2011-01-01', 500),
( 2, '2011-01-15', 50),
( 2, '2011-01-22', 25),
( 2, '2011-01-23', 125),
( 2, '2011-01-26', 200),
( 2, '2011-01-29', 250),
( 3, '2011-01-01', 500),
( 3, '2011-01-15', 50 ),
( 3, '2011-01-22', 5000),
( 3, '2011-01-25', 550),
( 3, '2011-01-27', 95 ),
( 3, '2011-01-30', 2500)



num_affected_rows,num_inserted_rows
18,18


In [0]:
Transactions = [( 1, '2011-01-01', 500),
( 1, '2011-01-15', 50),
( 1, '2011-01-22', 250),
( 1, '2011-01-24', 75),
( 1, '2011-01-26', 125),
( 1, '2011-01-28', 175),
( 2, '2011-01-01', 500),
( 2, '2011-01-15', 50),
( 2, '2011-01-22', 25),
( 2, '2011-01-23', 125),
( 2, '2011-01-26', 200),
( 2, '2011-01-29', 250),
( 3, '2011-01-01', 500),
( 3, '2011-01-15', 50 ),
( 3, '2011-01-22', 5000),
( 3, '2011-01-25', 550),
( 3, '2011-01-27', 95 ),
( 3, '2011-01-30', 2500)
]
transactions2_df = spark.createDataFrame(Transactions,["AccountId", "TranDate", "TranAmt"])
transactions2_df = transactions2_df.withColumn("TranDate", to_date(col("TranDate"), "yyyy-MM-dd"))

In [0]:
%sql
INSERT INTO Sample.Logical
VALUES (1,'George', 800),
(2,'Sam', 950),
(3,'Diane', 1100),
(4,'Nicholas', 1250),
(5,'Samuel', 1250),
(6,'Patricia', 1300),
(7,'Brian', 1500),
(8,'Thomas', 1600),
(9,'Fran', 2450),
(10,'Debbie', 2850),
(11,'Mark', 2975),
(12,'James', 3000),
(13,'Cynthia', 3000),
(14,'Christopher', 5000);

num_affected_rows,num_inserted_rows
14,14


In [0]:
Logical = [(1,'George', 800),
(2,'Sam', 950),
(3,'Diane', 1100),
(4,'Nicholas', 1250),
(5,'Samuel', 1250),
(6,'Patricia', 1300),
(7,'Brian', 1500),
(8,'Thomas', 1600),
(9,'Fran', 2450),
(10,'Debbie', 2850),
(11,'Mark', 2975),
(12,'James', 3000),
(13,'Cynthia', 3000),
(14,'Christopher', 5000)]
logical_df = spark.createDataFrame(Logical, ["RowID","FName", "Salary"])

Totals based on previous row

In [0]:
%sql

SELECT AccountId,
TranDate,
TranAmt,
-- running total of all transactions
SUM(TranAmt) OVER (PARTITION BY AccountId ORDER BY TranDate) as RunTotalAmt
FROM Sample.Transactions ORDER BY AccountId, TranDate;

AccountId,TranDate,TranAmt,RunTotalAmt
1,2011-01-01,500.00,1000.00
1,2011-01-01,500.00,1000.00
1,2011-01-15,50.00,1100.00
1,2011-01-15,50.00,1100.00
1,2011-01-22,250.00,1600.00
1,2011-01-22,250.00,1600.00
1,2011-01-24,75.00,1750.00
1,2011-01-24,75.00,1750.00
1,2011-01-26,125.00,2000.00
1,2011-01-26,125.00,2000.00


In [0]:
window1 = Window.partitionBy('AccountId').orderBy('TranDate')
transactions_df = transactions_df.withColumn('RunTotalAmt', sum('TranAmt').over(window1))\
  .orderBy('AccountId','TranDate')


In [0]:
%sql
SELECT AccountId,
TranDate,
TranAmt,
-- running average of all transactions
AVG(TranAmt) OVER (PARTITION BY AccountId ORDER BY TranDate) as RunAvg,
-- running total # of transactions
COUNT(*) OVER (PARTITION BY AccountId ORDER BY TranDate) as RunTranQty,
-- smallest of the transactions so far
MIN(TranAmt) OVER (PARTITION BY AccountId ORDER BY TranDate) as RunSmallAmt,
-- largest of the transactions so far
MAX(TranAmt) OVER (PARTITION BY AccountId ORDER BY TranDate) as RunLargeAmt,
-- running total of all transactions
SUM(TranAmt) OVER (PARTITION BY AccountId ORDER BY TranDate) RunTotalAmt
FROM Sample.Transactions 
ORDER BY AccountId,TranDate;

AccountId,TranDate,TranAmt,RunAvg,RunTranQty,RunSmallAmt,RunLargeAmt,RunTotalAmt
1,2011-01-01,500.00,500.000000,2,500.00,500.00,1000.00
1,2011-01-01,500.00,500.000000,2,500.00,500.00,1000.00
1,2011-01-15,50.00,275.000000,4,50.00,500.00,1100.00
1,2011-01-15,50.00,275.000000,4,50.00,500.00,1100.00
1,2011-01-22,250.00,266.666667,6,50.00,500.00,1600.00
1,2011-01-22,250.00,266.666667,6,50.00,500.00,1600.00
1,2011-01-24,75.00,218.750000,8,50.00,500.00,1750.00
1,2011-01-24,75.00,218.750000,8,50.00,500.00,1750.00
1,2011-01-26,125.00,200.000000,10,50.00,500.00,2000.00
1,2011-01-26,125.00,200.000000,10,50.00,500.00,2000.00


In [0]:
transactions_df = transactions_df.withColumn('RunaVG', avg('TranAmt').over(window1))\
  .orderBy('AccountId','TranDate')
transactions_df = transactions_df.withColumn('RunTranQty', count('*').over(window1))\
  .orderBy('AccountId','TranDate')
transactions_df = transactions_df.withColumn('RunSmallAmt', min('TranAmt').over(window1))\
  .orderBy('AccountId','TranDate') 
transactions_df = transactions_df.withColumn('RunLargeAmt', max('TranAmt').over(window1))\
  .orderBy('AccountId','TranDate') 
display(transactions_df.orderBy('AccountId','TranDate'))
#tamten notatnik z podwojnymi rekordami najprawdopodbniej 

AccountId,TranDate,TranAmt,RunTotalAmt,RunaVG,RunTranQty,RunSmallAmt,RunLargeAmt
1,2011-01-01,500,500,500.0,1,500,500
1,2011-01-15,50,550,275.0,2,50,500
1,2011-01-22,250,800,266.6666666666667,3,50,500
1,2011-01-24,75,875,218.75,4,50,500
1,2011-01-26,125,1000,200.0,5,50,500
1,2011-01-28,175,1175,195.83333333333334,6,50,500
2,2011-01-01,500,500,500.0,1,500,500
2,2011-01-15,50,550,275.0,2,50,500
2,2011-01-22,25,575,191.66666666666666,3,25,500
2,2011-01-23,125,700,175.0,4,25,500


* Calculating Totals Based Upon a Subset of Rows

In [0]:
%sql
SELECT AccountId,
TranDate,
TranAmt,
-- average of the current and previous 2 transactions
AVG(TranAmt) OVER (PARTITION BY AccountId ORDER BY TranDate ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as SlideAvg,
-- total # of the current and previous 2 transactions
COUNT(*) OVER (PARTITION BY AccountId ORDER BY TranDate ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as SlideQty,
-- smallest of the current and previous 2 transactions
MIN(TranAmt) OVER (PARTITION BY AccountId ORDER BY TranDate ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as SlideMin,
-- largest of the current and previous 2 transactions
MAX(TranAmt) OVER (PARTITION BY AccountId ORDER BY TranDate ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as SlideMax,
-- total of the current and previous 2 transactions
SUM(TranAmt) OVER (PARTITION BY AccountId ORDER BY TranDate ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as SlideTotal,
ROW_NUMBER() OVER (PARTITION BY AccountId ORDER BY TranDate) AS RN
FROM Sample.Transactions 
ORDER BY AccountId, TranDate, RN

AccountId,TranDate,TranAmt,SlideAvg,SlideQty,SlideMin,SlideMax,SlideTotal,RN
1,2011-01-01,500.00,500.000000,1,500.00,500.00,500.00,1
1,2011-01-01,500.00,500.000000,2,500.00,500.00,1000.00,2
1,2011-01-15,50.00,350.000000,3,50.00,500.00,1050.00,3
1,2011-01-15,50.00,200.000000,3,50.00,500.00,600.00,4
1,2011-01-22,250.00,116.666667,3,50.00,250.00,350.00,5
1,2011-01-22,250.00,183.333333,3,50.00,250.00,550.00,6
1,2011-01-24,75.00,191.666667,3,75.00,250.00,575.00,7
1,2011-01-24,75.00,133.333333,3,75.00,250.00,400.00,8
1,2011-01-26,125.00,91.666667,3,75.00,125.00,275.00,9
1,2011-01-26,125.00,108.333333,3,75.00,125.00,325.00,10


In [0]:
# podzial na id, sortowanie od daty, grupowanie po 3, aktualny wiesszz i dwa nad.
window2 = Window.partitionBy('AccountId').orderBy('TranDate').rowsBetween(-2,0) 
transactions_df = transactions_df.withColumn('SlideaVG', avg('TranAmt').over(window2))\
  .orderBy('AccountId')
transactions_df = transactions_df.withColumn('SlideTranQty', count('*').over(window2))\
  .orderBy('AccountId')
transactions_df = transactions_df.withColumn('SlideSmallAmt', min('TranAmt').over(window2))\
  .orderBy('AccountId') 
transactions_df = transactions_df.withColumn('SlideLargeAmt', max('TranAmt').over(window2))\
  .orderBy('AccountId') 
transactions_df = transactions_df.withColumn('SlideTotalAmt', sum('TranAmt').over(window2))\
  .orderBy('AccountId')
transactions_df = transactions_df.withColumn('RN', row_number().over(Window.partitionBy("AccountId").orderBy("TranDate")))\
  .orderBy('AccountId')
transactions_df.orderBy('AccountId','TranDate','RN').show()

+---------+----------+-------+------------------+------------+-------------+-------------+-------------+---+
|AccountId|  TranDate|TranAmt|          SlideaVG|SlideTranQty|SlideSmallAmt|SlideLargeAmt|SlideTotalAmt| RN|
+---------+----------+-------+------------------+------------+-------------+-------------+-------------+---+
|        1|2011-01-01|    500|             500.0|           1|          500|          500|          500|  1|
|        1|2011-01-15|     50|             275.0|           2|           50|          500|          550|  2|
|        1|2011-01-22|    250| 266.6666666666667|           3|           50|          500|          800|  3|
|        1|2011-01-24|     75|             125.0|           3|           50|          250|          375|  4|
|        1|2011-01-26|    125|             150.0|           3|           75|          250|          450|  5|
|        1|2011-01-28|    175|             125.0|           3|           75|          175|          375|  6|
|        2|2011-01-

* Logical Window


In [0]:
%sql
SELECT RowID,
FName,
Salary,
SUM(Salary) OVER (ORDER BY Salary ROWS UNBOUNDED PRECEDING) as SumByRows,
SUM(Salary) OVER (ORDER BY Salary RANGE UNBOUNDED PRECEDING) as SumByRange,

FROM Sample.Logical
ORDER BY RowID;

RowID,FName,Salary,SumByRows,SumByRange
1,George,800,800,800
2,Sam,950,1750,1750
3,Diane,1100,2850,2850
4,Nicholas,1250,4100,5350
5,Samuel,1250,5350,5350
6,Patricia,1300,6650,6650
7,Brian,1500,8150,8150
8,Thomas,1600,9750,9750
9,Fran,2450,12200,12200
10,Debbie,2850,15050,15050


In [0]:
# suma do aktualnego okienka posortwana po salary
window3 = Window.orderBy("Salary").rowsBetween(Window.unboundedPreceding, Window.currentRow)
window4 = Window.orderBy("Salary").rangeBetween(Window.unboundedPreceding, Window.currentRow)
logical_df = logical_df.withColumn("SumByRows", sum('Salary').over(window3))
logical_df = logical_df.withColumn("SumByRange", sum('Salary').over(window4))
logical_df.orderBy('RowID').show()

+-----+-----------+------+---------+----------+
|RowID|      FName|Salary|SumByRows|SumByRange|
+-----+-----------+------+---------+----------+
|    1|     George|   800|      800|       800|
|    2|        Sam|   950|     1750|      1750|
|    3|      Diane|  1100|     2850|      2850|
|    4|   Nicholas|  1250|     4100|      5350|
|    5|     Samuel|  1250|     5350|      5350|
|    6|   Patricia|  1300|     6650|      6650|
|    7|      Brian|  1500|     8150|      8150|
|    8|     Thomas|  1600|     9750|      9750|
|    9|       Fran|  2450|    12200|     12200|
|   10|     Debbie|  2850|    15050|     15050|
|   11|       Mark|  2975|    18025|     18025|
|   12|      James|  3000|    21025|     24025|
|   13|    Cynthia|  3000|    24025|     24025|
|   14|Christopher|  5000|    29025|     29025|
+-----+-----------+------+---------+----------+



In [0]:
transactions2_df = transactions2_df.withColumn("LeadValue", lead("TranAmt").over(window1))
transactions2_df = transactions2_df.withColumn("LagValue", lag("TranAmt").over(window1))
transactions2_df = transactions2_df.withColumn("FirstValue", first("TranAmt").over(window1))
transactions2_df = transactions2_df.withColumn("LastValue", last("TranAmt").over(window1))
transactions2_df = transactions2_df.withColumn("RowNumber", row_number().over(window1))
display(transactions2_df.orderBy('AccountId','TranDate').show() )

+---------+----------+-------+---------+--------+----------+---------+---------+
|AccountId|  TranDate|TranAmt|LeadValue|LagValue|FirstValue|LastValue|RowNumber|
+---------+----------+-------+---------+--------+----------+---------+---------+
|        1|2011-01-01|    500|       50|    null|       500|      500|        1|
|        1|2011-01-15|     50|      250|     500|       500|       50|        2|
|        1|2011-01-22|    250|       75|      50|       500|      250|        3|
|        1|2011-01-24|     75|      125|     250|       500|       75|        4|
|        1|2011-01-26|    125|      175|      75|       500|      125|        5|
|        1|2011-01-28|    175|     null|     125|       500|      175|        6|
|        2|2011-01-01|    500|       50|    null|       500|      500|        1|
|        2|2011-01-15|     50|       25|     500|       500|       50|        2|
|        2|2011-01-22|     25|      125|      50|       500|       25|        3|
|        2|2011-01-23|    12